In [0]:
from pyspark.sql.functions import *

# Read Bronze JSON Table
json_df = spark.table("workauto.bronze.json_raw")

# Remove Duplicate Records
json_df = json_df.dropDuplicates(["product_id"])


# Trim Spaces
json_df = (
    json_df
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("supplier", trim(col("supplier")))
)

# Standardize Text Columns
json_df = (
    json_df
    .withColumn("product_name", initcap(col("product_name")))
    .withColumn("category", upper(col("category")))
    .withColumn("supplier", initcap(col("supplier")))
)

# Handle Null Product Names
json_df = json_df.fillna({
    "product_name": "Unknown Product",
    "supplier": "Unknown Supplier"
})

# Replace Empty Categories
json_df = json_df.withColumn(
    "category",
    when(col("category") == "", "UNKNOWN")
    .otherwise(col("category"))
)

# Remove Invalid Stock
json_df = json_df.filter(col("stock") >= 0)

# Inventory Status Logic
json_df = json_df.withColumn(
    "inventory_status",
    when(col("stock") == 0, "Out of Stock")
    .when(col("stock") < 50, "Low Stock")
    .otherwise("Available")
)

# Add Workflow Action
json_df = json_df.withColumn(
    "workflow_action",
    when(col("inventory_status") == "Low Stock", "Reorder Required")
    .when(col("inventory_status") == "Out of Stock", "Urgent Restock")
    .otherwise("No Action")
)

# Add Process Timestamp
json_df = json_df.withColumn(
    "processed_time",
    current_timestamp()
)

# Show Data
json_df.show(truncate=False)

# Save Silver Table
(
    json_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workauto.silver.json_clean")
)

print("JSON Silver Table Created Successfully")